# SupplyChain IQ — Step 1: Data Quality & EDA

Dataset note: these are synthetic practice data, not real Philips or company data.

## Learning goals
- Load CSV files with Pandas
- Inspect columns, types, and missing values
- Check duplicates and invalid values
- Define initial supply-chain KPIs


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# If running from this notebook in the project folder, this path should work.
DATA = Path('../data/raw')
orders = pd.read_csv(DATA / 'orders.csv', parse_dates=['order_date', 'promised_delivery_date', 'actual_delivery_date'])
products = pd.read_csv(DATA / 'products.csv')
suppliers = pd.read_csv(DATA / 'suppliers.csv')
inventory = pd.read_csv(DATA / 'inventory.csv', parse_dates=['snapshot_date'])

print('Orders:', orders.shape)
print('Products:', products.shape)
print('Suppliers:', suppliers.shape)
print('Inventory:', inventory.shape)


Orders: (2200, 9)
Products: (15, 5)
Suppliers: (5, 4)
Inventory: (15, 4)


In [2]:
# 1. Check duplicate order IDs
print("Duplicate order IDs:",
      orders["order_id"].duplicated().sum())

# 2. Check missing values
print("\nMissing values:")
print(orders.isnull().sum())

# 3. Check invalid quantities
print("\nZero or negative quantities:",
      (orders["quantity"].dropna() <= 0).sum())

# 4. Check order statuses
print("\nOrder status counts:")
print(orders["order_status"].value_counts(dropna=False))

Duplicate order IDs: 0

Missing values:
order_id                   0
order_date                 0
product_id                 0
supplier_id               20
quantity                  35
unit_price                 0
promised_delivery_date     0
actual_delivery_date       0
order_status               0
dtype: int64

Zero or negative quantities: 0

Order status counts:
order_status
Delivered     1826
In Transit     259
Cancelled      115
Name: count, dtype: int64


In [3]:
# Create a working copy; preserve the original dataset
orders_clean = orders.copy()

# Remove rows only if the order ID is missing
orders_clean = orders_clean.dropna(subset=["order_id"])

# Check missing values after the initial cleaning
print("Rows before cleaning:", len(orders))
print("Rows after initial cleaning:", len(orders_clean))

print("\nMissing values after initial cleaning:")
print(orders_clean.isna().sum())

Rows before cleaning: 2200
Rows after initial cleaning: 2200

Missing values after initial cleaning:
order_id                   0
order_date                 0
product_id                 0
supplier_id               20
quantity                  35
unit_price                 0
promised_delivery_date     0
actual_delivery_date       0
order_status               0
dtype: int64


In [4]:
# Calculate delivery delay in days
orders_clean["delivery_delay_days"] = (
    orders_clean["actual_delivery_date"]
    - orders_clean["promised_delivery_date"]
).dt.days

# Calculate order value
orders_clean["order_value"] = (
    orders_clean["quantity"] * orders_clean["unit_price"]
)

# Identify on-time delivered orders
orders_clean["on_time"] = (
    (orders_clean["order_status"] == "Delivered")
    & (orders_clean["delivery_delay_days"] <= 0)
)

# Calculate KPIs for delivered orders only
delivered = orders_clean[
    orders_clean["order_status"] == "Delivered"
]

on_time_rate = delivered["on_time"].mean() * 100

print("Total order records:", len(orders_clean))
print("Delivered orders:", len(delivered))
print("On-time deliveries:", int(delivered["on_time"].sum()))
print("On-time delivery rate:", round(on_time_rate, 2), "%")
print("Total order value (known quantities):",
      round(orders_clean["order_value"].sum(), 2))
print("Missing order values:",
      orders_clean["order_value"].isna().sum())

Total order records: 2200
Delivered orders: 1826
On-time deliveries: 859
On-time delivery rate: 47.04 %
Total order value (known quantities): 42140877.79
Missing order values: 35


In [5]:
# Combine inventory information with product details
inventory_analysis = inventory.merge(
    products,
    on="product_id",
    how="left"
)

# Identify products below their reorder level
inventory_analysis["stock_status"] = np.where(
    inventory_analysis["stock_on_hand"]
    < inventory_analysis["reorder_level"],
    "Low Stock",
    "Sufficient Stock"
)

# Display low-stock products
low_stock = inventory_analysis[
    inventory_analysis["stock_status"] == "Low Stock"
]

print("Total products:", len(inventory_analysis))
print("Products with low stock:", len(low_stock))

display(
    low_stock[
        ["product_id", "product_name", "stock_on_hand", "reorder_level"]
    ].sort_values("stock_on_hand")
)

Total products: 15
Products with low stock: 6


,product_id,product_name,stock_on_hand,reorder_level
13,P014,Mounting Bracket,6,48
1,P002,Sensor Module,12,38
9,P010,Filter Cartridge,23,70
7,P008,Display Unit,44,86
4,P005,Valve Assembly,63,91
6,P007,Pump Component,71,88


In [6]:
# Keep delivered orders with known supplier IDs
supplier_orders = orders_clean[
    (orders_clean["order_status"] == "Delivered")
    & (orders_clean["supplier_id"].notna())
].copy()

# Calculate supplier-level delivery metrics
supplier_performance = (
    supplier_orders.groupby("supplier_id")
    .agg(
        delivered_orders=("order_id", "count"),
        average_delay_days=("delivery_delay_days", "mean"),
        on_time_deliveries=("on_time", "sum")
    )
    .reset_index()
)

supplier_performance["on_time_rate_pct"] = (
    supplier_performance["on_time_deliveries"]
    / supplier_performance["delivered_orders"] * 100
)

# Add supplier names and regions
supplier_performance = supplier_performance.merge(
    suppliers[["supplier_id", "supplier_name", "supplier_region"]],
    on="supplier_id",
    how="left"
)

supplier_performance = supplier_performance.sort_values(
    "on_time_rate_pct"
)

display(supplier_performance.round(2))

,supplier_id,delivered_orders,average_delay_days,on_time_deliveries,on_time_rate_pct,supplier_name,supplier_region
0,S01,367,1.94,155,42.23,Northstar Components,South
2,S03,364,1.97,159,43.68,MedTech Materials,North
3,S04,368,1.80,181,49.18,Apex Industrial,East
4,S05,354,1.79,176,49.72,BlueWave Logistics,Central
1,S02,357,1.78,178,49.86,Precision Supply Co,West


In [7]:
display(
    supplier_performance[
        [
            "supplier_name",
            "supplier_region",
            "delivered_orders",
            "average_delay_days",
            "on_time_rate_pct"
        ]
    ].round(2)
)

,supplier_name,supplier_region,delivered_orders,average_delay_days,on_time_rate_pct
0,Northstar Components,South,367,1.94,42.23
2,MedTech Materials,North,364,1.97,43.68
3,Apex Industrial,East,368,1.80,49.18
4,BlueWave Logistics,Central,354,1.79,49.72
1,Precision Supply Co,West,357,1.78,49.86


In [8]:
display(
    supplier_performance[
        [
            "supplier_name",
            "delivered_orders",
            "average_delay_days",
            "on_time_deliveries",
            "on_time_rate_pct"
        ]
    ].round(2).to_string(index=False)
)

'       supplier_name  delivered_orders  average_delay_days  on_time_deliveries  on_time_rate_pct\nNorthstar Components               367                1.94                 155             42.23\n   MedTech Materials               364                1.97                 159             43.68\n     Apex Industrial               368                1.80                 181             49.18\n  BlueWave Logistics               354                1.79                 176             49.72\n Precision Supply Co               357                1.78                 178             49.86'

In [9]:
supplier_view = supplier_performance[
    [
        "supplier_name",
        "delivered_orders",
        "average_delay_days",
        "on_time_deliveries",
        "on_time_rate_pct"
    ]
].round(2)

print(supplier_view.to_string(index=False))

       supplier_name  delivered_orders  average_delay_days  on_time_deliveries  on_time_rate_pct
Northstar Components               367                1.94                 155             42.23
   MedTech Materials               364                1.97                 159             43.68
     Apex Industrial               368                1.80                 181             49.18
  BlueWave Logistics               354                1.79                 176             49.72
 Precision Supply Co               357                1.78                 178             49.86


In [10]:
print(
    supplier_performance[
        ["supplier_name", "on_time_deliveries", "on_time_rate_pct"]
    ].round(2).to_string(index=False)
)

       supplier_name  on_time_deliveries  on_time_rate_pct
Northstar Components                 155             42.23
   MedTech Materials                 159             43.68
     Apex Industrial                 181             49.18
  BlueWave Logistics                 176             49.72
 Precision Supply Co                 178             49.86


In [11]:
import sqlite3
import pandas as pd

# Create a local SQL database
conn = sqlite3.connect("supplychain.db")

# Load the datasets into SQL tables
orders_clean.to_sql(
    "orders", conn, if_exists="replace", index=False
)

products.to_sql(
    "products", conn, if_exists="replace", index=False
)

suppliers.to_sql(
    "suppliers", conn, if_exists="replace", index=False
)

inventory.to_sql(
    "inventory", conn, if_exists="replace", index=False
)

print("All four datasets loaded into SQL tables!")

All four datasets loaded into SQL tables!


In [12]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,order_status,total_orders
0,Delivered,1826
1,In Transit,259
2,Cancelled,115


In [13]:
query = """
SELECT
    s.supplier_name,
    COUNT(*) AS delivered_orders,
    ROUND(AVG(o.delivery_delay_days), 2)
        AS average_delay_days,
    ROUND(100.0 * AVG(o.on_time), 2)
        AS on_time_rate_pct
FROM orders AS o
JOIN suppliers AS s
    ON o.supplier_id = s.supplier_id
WHERE o.order_status = 'Delivered'
GROUP BY s.supplier_id, s.supplier_name
ORDER BY on_time_rate_pct DESC;
"""

supplier_sql = pd.read_sql_query(query, conn)
display(supplier_sql)

,supplier_name,delivered_orders,average_delay_days,on_time_rate_pct
0,Precision Supply Co,357,1.78,49.86
1,BlueWave Logistics,354,1.79,49.72
2,Apex Industrial,368,1.80,49.18
3,MedTech Materials,364,1.97,43.68
4,Northstar Components,367,1.94,42.23


In [14]:
query = """
SELECT
    p.product_id,
    p.product_name,
    i.stock_on_hand,
    p.reorder_level,
    p.reorder_level - i.stock_on_hand AS units_below_reorder
FROM inventory AS i
JOIN products AS p
    ON i.product_id = p.product_id
WHERE i.stock_on_hand < p.reorder_level
ORDER BY units_below_reorder DESC;
"""

low_stock_sql = pd.read_sql_query(query, conn)
display(low_stock_sql)

,product_id,product_name,stock_on_hand,reorder_level,units_below_reorder
0,P010,Filter Cartridge,23,70,47
1,P008,Display Unit,44,86,42
2,P014,Mounting Bracket,6,48,42
3,P005,Valve Assembly,63,91,28
4,P002,Sensor Module,12,38,26
5,P007,Pump Component,71,88,17


In [15]:
query = """
SELECT
    p.category,
    COUNT(o.order_id) AS total_orders,
    ROUND(
        SUM(o.quantity * o.unit_price), 2
    ) AS total_order_value
FROM orders AS o
JOIN products AS p
    ON o.product_id = p.product_id
WHERE o.quantity IS NOT NULL
  AND o.order_status != 'Cancelled'
GROUP BY p.category
ORDER BY total_order_value DESC;
"""

category_analysis = pd.read_sql_query(query, conn)
display(category_analysis)

,category,total_orders,total_order_value
0,Components,856,16691032.91
1,Electronics,515,9763089.18
2,Electrical,403,7810426.94
3,Consumables,153,2923224.38
4,Packaging,124,2852636.72


In [16]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("supplychain.db")

query = """
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE
        WHEN quantity IS NULL THEN 1
        ELSE 0
    END) AS missing_quantities,
    SUM(CASE
        WHEN supplier_id IS NULL THEN 1
        ELSE 0
    END) AS missing_suppliers
FROM orders;
"""

display(pd.read_sql_query(query, conn))

,total_orders,missing_quantities,missing_suppliers
0,2200,35,20
